# Task 1 - Business Sales Performance Analytics
**Future Interns | Data Science & Analytics**
**Auteur :** DOBARAPIENA DOMINIQUE TUO | **CIN :** FIT/APR26/DS17023  
**Repository GitHub :** FUTURE_DS_01

---
## Objectif
Analyser les donnees de ventes pour identifier les tendances de revenus, les produits stars, les meilleures regions et les performances des vendeurs.

## 1. Chargement des bibliotheques et des donnees

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "#F8F9FA"
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print("Bibliotheques chargees avec succes")

In [ ]:
df = pd.read_csv("dataset_task1_sales.csv")
df["Date"] = pd.to_datetime(df["Date"])
print(f"Dataset charge : {df.shape[0]} lignes, {df.shape[1]} colonnes")
df.head()

## 2. Exploration et Nettoyage des Donnees

In [ ]:
print("=== STRUCTURE ===")
print(df.dtypes)
print(f"
Valeurs manquantes : {df.isnull().sum().sum()}")
print(f"Doublons : {df.duplicated().sum()}")
df.describe().round(0)

## 3. Indicateurs Cles de Performance (KPIs)

In [ ]:
total_revenu    = df["Revenu_FCFA"].sum()
total_profit    = df["Profit_FCFA"].sum()
marge_globale   = total_profit / total_revenu * 100
nb_transactions = len(df)
total_unites    = df["Quantite"].sum()
panier_moyen    = total_revenu / nb_transactions

print("=" * 50)
print("        TABLEAU DE BORD KPIs 2024")
print("=" * 50)
print(f"  Revenu Total      : {total_revenu:>16,.0f} FCFA")
print(f"  Profit Total      : {total_profit:>16,.0f} FCFA")
print(f"  Marge Beneficiaire: {marge_globale:>15.1f} %")
print(f"  Nb Transactions   : {nb_transactions:>16,}")
print(f"  Unites Vendues    : {total_unites:>16,}")
print(f"  Panier Moyen      : {panier_moyen:>16,.0f} FCFA")
print("=" * 50)

## 4. Visualisations

### 4.1 - Top Produits par Revenu et Profit

In [ ]:
prod = df.groupby("Produit").agg(
    Revenu=("Revenu_FCFA","sum"),
    Profit=("Profit_FCFA","sum"),
    Quantite=("Quantite","sum")
).sort_values("Revenu", ascending=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("Analyse des Ventes par Produit", fontsize=16, fontweight="bold", color="#1F4E79")

colors_rev = ["#1F4E79" if i==len(prod)-1 else "#2E75B6" if i>=len(prod)-3 else "#BDD7EE" for i in range(len(prod))]
bars = ax1.barh(prod.index, prod["Revenu"]/1e6, color=colors_rev, edgecolor="white")
ax1.set_xlabel("Revenu (millions FCFA)")
ax1.set_title("Revenu Total par Produit", fontweight="bold", color="#1F4E79")
for bar, val in zip(bars, prod["Revenu"]):
    ax1.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2, f"{val/1e6:.1f}M", va="center", fontsize=9, fontweight="bold")

prod_s = prod.sort_values("Revenu", ascending=False)
x = range(len(prod_s))
ax2.bar([i-0.2 for i in x], prod_s["Revenu"]/1e6, 0.4, label="Revenu", color="#2E75B6")
ax2.bar([i+0.2 for i in x], prod_s["Profit"]/1e6, 0.4, label="Profit", color="#1E7145")
ax2.set_xticks(list(x))
ax2.set_xticklabels([p[:10] for p in prod_s.index], rotation=30, ha="right")
ax2.set_title("Revenu vs Profit par Produit", fontweight="bold", color="#1F4E79")
ax2.legend()

plt.tight_layout()
plt.savefig("graph1_produits.png", dpi=150, bbox_inches="tight")
plt.show()
print("Graph 1 sauvegarde")

### 4.2 - Performance par Region

In [ ]:
region = df.groupby("Region").agg(
    Revenu=("Revenu_FCFA","sum"),
    Profit=("Profit_FCFA","sum"),
    Transactions=("Revenu_FCFA","count")
).sort_values("Revenu", ascending=False).reset_index()
region["Marge"] = (region["Profit"]/region["Revenu"]*100).round(1)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Performance des Ventes par Region", fontsize=16, fontweight="bold", color="#1E7145")
palette_g = ["#1E7145","#2E8B57","#3CB371","#52B788","#74C69D","#95D5B2"]

axes[0].bar(region["Region"], region["Revenu"]/1e6, color=palette_g, edgecolor="white")
axes[0].set_title("Revenu par Region", fontweight="bold")
axes[0].set_xticklabels(region["Region"], rotation=25, ha="right")
for i, r in enumerate(region["Revenu"]):
    axes[0].text(i, r/1e6+0.5, f"{r/1e6:.1f}M", ha="center", fontsize=8, fontweight="bold")

axes[1].pie(region["Revenu"], labels=region["Region"], autopct="%1.1f%%", colors=palette_g,
            startangle=90, wedgeprops=dict(edgecolor="white", linewidth=2))
axes[1].set_title("Part de Marche par Region", fontweight="bold")

axes[2].bar(region["Region"], region["Marge"], color="#95D5B2", edgecolor="white")
axes[2].set_title("Marge par Region (%)", fontweight="bold")
axes[2].set_xticklabels(region["Region"], rotation=25, ha="right")
for i, m in enumerate(region["Marge"]):
    axes[2].text(i, m+0.2, f"{m}%", ha="center", fontsize=9, fontweight="bold", color="#1E7145")

plt.tight_layout()
plt.savefig("graph2_regions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Graph 2 sauvegarde")

### 4.3 - Evolution Temporelle

In [ ]:
mois_order = ["January","February","March","April","May","June",
              "July","August","September","October","November","December"]
mois_fr = ["Jan","Fev","Mar","Avr","Mai","Jun","Jul","Aou","Sep","Oct","Nov","Dec"]
df["Mois_Num"] = pd.Categorical(df["Mois"], categories=mois_order, ordered=True)
monthly = df.groupby("Mois_Num", observed=True).agg(
    Revenu=("Revenu_FCFA","sum"),
    Profit=("Profit_FCFA","sum"),
    Trans=("Revenu_FCFA","count")
).reset_index()
monthly["MoisFR"] = mois_fr[:len(monthly)]

trimestriel = df.groupby("Trimestre").agg(Revenu=("Revenu_FCFA","sum"),Profit=("Profit_FCFA","sum")).reset_index()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Evolution Temporelle des Ventes 2024", fontsize=16, fontweight="bold", color="#1F4E79")

axes[0,0].plot(monthly["MoisFR"], monthly["Revenu"]/1e6, marker="o", color="#1F4E79", lw=2.5, ms=7, mfc="white", mew=2)
axes[0,0].fill_between(range(len(monthly)), monthly["Revenu"]/1e6, alpha=0.15, color="#2E75B6")
axes[0,0].set_title("Revenu Mensuel (M FCFA)", fontweight="bold", color="#1F4E79")
axes[0,0].set_xticks(range(len(monthly)))
axes[0,0].set_xticklabels(monthly["MoisFR"], rotation=30)

axes[0,1].bar(monthly["MoisFR"], monthly["Profit"]/1e6, color=["#1E7145" if v>=monthly["Profit"].mean() else "#95D5B2" for v in monthly["Profit"]])
axes[0,1].axhline(monthly["Profit"].mean()/1e6, color="#C55A11", ls="--", lw=1.5, label=f"Moy: {monthly["Profit"].mean()/1e6:.0f}M")
axes[0,1].set_title("Profit Mensuel (M FCFA)", fontweight="bold", color="#1E7145")
axes[0,1].set_xticklabels(monthly["MoisFR"], rotation=30)
axes[0,1].legend()

axes[1,0].bar(monthly["MoisFR"], monthly["Trans"], color="#7030A0", alpha=0.8)
axes[1,0].set_title("Transactions par Mois", fontweight="bold", color="#7030A0")
axes[1,0].set_xticklabels(monthly["MoisFR"], rotation=30)

axes[1,1].bar(["T1","T2","T3","T4"], trimestriel["Revenu"]/1e6, color=["#1F4E79","#2E75B6","#BDD7EE","#D6E4F0"], width=0.5)
axes[1,1].set_title("Revenu par Trimestre (M FCFA)", fontweight="bold", color="#1F4E79")

plt.tight_layout()
plt.savefig("graph3_temporel.png", dpi=150, bbox_inches="tight")
plt.show()
print("Graph 3 sauvegarde")

### 4.4 - Heatmap Region x Categorie

In [ ]:
pivot = df.pivot_table(values="Revenu_FCFA", index="Region", columns="Categorie", aggfunc="sum", fill_value=0)
fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot/1e6, annot=True, fmt=".1f", cmap="Blues", linewidths=0.5, linecolor="white",
            ax=ax, cbar_kws={"label": "Revenu (M FCFA)"})
ax.set_title("Heatmap : Revenu par Region x Categorie (M FCFA)", fontsize=14, fontweight="bold", color="#1F4E79")
ax.set_xlabel("Categorie")
ax.set_ylabel("Region")
plt.tight_layout()
plt.savefig("graph4_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Graph 4 sauvegarde")

### 4.5 - Classement des Vendeurs

In [ ]:
vendeur = df.groupby("Vendeur").agg(
    Revenu=("Revenu_FCFA","sum"),
    Profit=("Profit_FCFA","sum"),
    Trans=("Revenu_FCFA","count")
).sort_values("Revenu", ascending=False).reset_index()
vendeur["Marge"] = (vendeur["Profit"]/vendeur["Revenu"]*100).round(1)

fig, ax = plt.subplots(figsize=(12, 6))
colors_v = ["#FFD700","#C0C0C0","#CD7F32"] + ["#7030A0"]*(len(vendeur)-3)
bars = ax.barh(vendeur["Vendeur"][::-1], vendeur["Revenu"][::-1]/1e6, color=colors_v[::-1], edgecolor="white")
ax.set_title("Classement des Vendeurs par Revenu", fontsize=14, fontweight="bold", color="#7030A0")
ax.set_xlabel("Revenu (M FCFA)")
for bar, val in zip(bars, vendeur["Revenu"][::-1]):
    ax.text(bar.get_width()+0.2, bar.get_y()+bar.get_height()/2, f"{val/1e6:.1f}M", va="center", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.savefig("graph5_vendeurs.png", dpi=150, bbox_inches="tight")
plt.show()
print("Graph 5 sauvegarde")

## 5. Insights Cles et Recommandations

In [ ]:
prod_top = df.groupby("Produit")["Revenu_FCFA"].sum().sort_values(ascending=False)
reg_top  = df.groupby("Region")["Revenu_FCFA"].sum().sort_values(ascending=False)
vend_top = df.groupby("Vendeur")["Revenu_FCFA"].sum().sort_values(ascending=False)

print("=" * 60)
print("   RAPPORT D INSIGHTS - BUSINESS SALES ANALYTICS 2024")
print("=" * 60)
print(f"")
print(f"INSIGHT 1 - PRODUIT STAR")
print(f"  -> {prod_top.index[0]} genere {prod_top.iloc[0]/1e6:.1f}M FCFA de revenu (N1)")
print(f"  RECOMMANDATION : Augmenter les stocks, creer des bundles.")
print(f"")
print(f"INSIGHT 2 - PERFORMANCE REGIONALE")
print(f"  -> {reg_top.index[0]} domine : {reg_top.iloc[0]/1e6:.1f}M FCFA")
print(f"  -> {reg_top.index[-1]} est derniere : {reg_top.iloc[-1]/1e6:.1f}M FCFA")
print(f"  RECOMMANDATION : Deployer les strategies gagnantes regionalement.")
print(f"")
print(f"INSIGHT 3 - MARGE GLOBALE")
print(f"  -> Marge : {marge_globale:.1f}% - surveiller les remises accordees.")
print(f"  RECOMMANDATION : Limiter les remises > 10% aux gros clients.")
print(f"")
print(f"INSIGHT 4 - MEILLEUR VENDEUR")
print(f"  -> {vend_top.index[0]} : {vend_top.iloc[0]/1e6:.1f}M FCFA (top performer)")
print(f"  RECOMMANDATION : Identifier et partager ses meilleures pratiques.")
print("=" * 60)

## 6. Conclusion

Cette analyse des ventes 2024 a permis d identifier :

| Priorite | Action | Impact |
|----------|--------|--------|
| Haute | Renforcer stocks du produit star | +15% revenu |
| Moyenne | Developper regions sous-performantes | +10% revenu |
| Normale | Former les vendeurs | +8% revenu |
| Continue | Anticiper la saisonnalite | Stabilite |

---
*Analyse realisee dans le cadre du stage Future Interns - Data Science & Analytics*  
*Repository : FUTURE_DS_01 | Auteur : DOBARAPIENA DOMINIQUE TUO*